In [6]:
import pandas as pd
import numpy as np
import os

raw_path = "../data/raw"
processed_path = "../data/processed"
os.makedirs(processed_path, exist_ok=True)

files = sorted([f for f in os.listdir(raw_path) if f.endswith(".csv")])
dfs = {}
for f in files:
    name = f.replace("olist_", "").replace("_dataset.csv", "").replace(".csv", "")
    dfs[name] = pd.read_csv(os.path.join(raw_path, f))

print("Loaded tables:", list(dfs.keys()))

Loaded tables: ['customers', 'geolocation', 'order_items', 'order_payments', 'order_reviews', 'orders', 'products', 'sellers', 'product_category_name_translation']


In [7]:
orders = dfs["orders"].copy()

date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

# Flag: was this order actually delivered
orders["is_delivered"] = orders["order_status"] == "delivered"

# Delivery time metrics (only meaningful where delivered)
orders["delivery_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"] - orders["order_estimated_delivery_date"]
).dt.days
# positive = late, negative = early

orders["approval_days"] = (
    orders["order_approved_at"] - orders["order_purchase_timestamp"]
).dt.days

# Calendar breakdowns
orders["order_year"] = orders["order_purchase_timestamp"].dt.year
orders["order_month"] = orders["order_purchase_timestamp"].dt.month
orders["order_year_month"] = orders["order_purchase_timestamp"].dt.to_period("M").astype(str)
orders["order_quarter"] = orders["order_purchase_timestamp"].dt.quarter
orders["order_dow"] = orders["order_purchase_timestamp"].dt.day_name()

print(orders[["order_status","is_delivered","delivery_days","delivery_delay_days"]].describe(include="all"))

       order_status is_delivered  delivery_days  delivery_delay_days
count         99441        99441   96476.000000         96476.000000
unique            8            2            NaN                  NaN
top       delivered         True            NaN                  NaN
freq          96478        96478            NaN                  NaN
mean            NaN          NaN      12.094086           -11.876881
std             NaN          NaN       9.551746            10.183854
min             NaN          NaN       0.000000          -147.000000
25%             NaN          NaN       6.000000           -17.000000
50%             NaN          NaN      10.000000           -12.000000
75%             NaN          NaN      15.000000            -7.000000
max             NaN          NaN     209.000000           188.000000


In [8]:
# Any negative delivery_days would mean delivered BEFORE purchase — impossible, check for it
print("Negative delivery_days count:", (orders["delivery_days"] < 0).sum())

# Distribution sanity check
print(orders["delivery_days"].describe())
print(orders["delivery_delay_days"].describe())

Negative delivery_days count: 0
count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64
count    96476.000000
mean       -11.876881
std         10.183854
min       -147.000000
25%        -17.000000
50%        -12.000000
75%         -7.000000
max        188.000000
Name: delivery_delay_days, dtype: float64


In [9]:
products = dfs["products"].copy()
cat_translation = dfs["product_category_name_translation"].copy()

# Merge English category names in now, so every downstream table can use english names
products = products.merge(cat_translation, on="product_category_name", how="left")

# The 610 rows missing category name — mark explicitly rather than silently dropping
products["product_category_name"] = products["product_category_name"].fillna("unknown")
products["product_category_name_english"] = products["product_category_name_english"].fillna("unknown")

# Weight/dimension: only 2 rows missing — impute with median (negligible impact either way)
for col in ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products[col] = products[col].fillna(products[col].median())

print(products.isnull().sum())
print(products["product_category_name_english"].value_counts().head(10))

product_id                         0
product_category_name              0
product_name_lenght              610
product_description_lenght       610
product_photos_qty               610
product_weight_g                   0
product_length_cm                  0
product_height_cm                  0
product_width_cm                   0
product_category_name_english      0
dtype: int64
product_category_name_english
bed_bath_table           3029
sports_leisure           2867
furniture_decor          2657
health_beauty            2444
housewares               2335
auto                     1900
computers_accessories    1639
toys                     1411
watches_gifts            1329
telephony                1134
Name: count, dtype: int64


In [10]:
geo = dfs["geolocation"].copy()

geo_clean = (
    geo.groupby("geolocation_zip_code_prefix")
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
        geolocation_state=("geolocation_state", lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan),
    )
    .reset_index()
)

print("Before:", len(geo), "After:", len(geo_clean))

Before: 1000163 After: 19015


In [11]:
payments = dfs["order_payments"].copy()
payments["payment_type"] = payments["payment_type"].replace("not_defined", "unknown")

print(payments["payment_type"].value_counts())

payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
unknown            3
Name: count, dtype: int64


In [12]:
reviews = dfs["order_reviews"].copy()

# Don't fill free text — just flag whether a comment exists, useful as a feature later
reviews["has_comment_message"] = reviews["review_comment_message"].notnull()
reviews["has_comment_title"] = reviews["review_comment_title"].notnull()

# Parse review dates
reviews["review_creation_date"] = pd.to_datetime(reviews["review_creation_date"], errors="coerce")
reviews["review_answer_timestamp"] = pd.to_datetime(reviews["review_answer_timestamp"], errors="coerce")

print(reviews[["review_score","has_comment_message","has_comment_title"]].describe(include="all"))

        review_score has_comment_message has_comment_title
count   99224.000000               99224             99224
unique           NaN                   2                 2
top              NaN               False             False
freq             NaN               58247             87656
mean        4.086421                 NaN               NaN
std         1.347579                 NaN               NaN
min         1.000000                 NaN               NaN
25%         4.000000                 NaN               NaN
50%         5.000000                 NaN               NaN
75%         5.000000                 NaN               NaN
max         5.000000                 NaN               NaN


In [13]:
order_items = dfs["order_items"].copy()
customers = dfs["customers"].copy()
sellers = dfs["sellers"].copy()

order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"], errors="coerce")

print("order_items rows:", len(order_items))
print("customers rows:", len(customers))
print("sellers rows:", len(sellers))

order_items rows: 112650
customers rows: 99441
sellers rows: 3095


In [14]:
# Total item value + freight per order, from order_items
order_item_totals = order_items.groupby("order_id").agg(
    total_items=("order_item_id", "count"),
    total_item_value=("price", "sum"),
    total_freight_value=("freight_value", "sum"),
).reset_index()

order_item_totals["order_total_value"] = (
    order_item_totals["total_item_value"] + order_item_totals["total_freight_value"]
)
order_item_totals["freight_pct"] = (
    order_item_totals["total_freight_value"] / order_item_totals["order_total_value"]
)

print(order_item_totals.describe())

        total_items  total_item_value  total_freight_value  order_total_value  \
count  98666.000000      98666.000000         98666.000000       98666.000000   
mean       1.141731        137.754076            22.823562         160.577638   
std        0.538452        210.645145            21.650909         220.466087   
min        1.000000          0.850000             0.000000           9.590000   
25%        1.000000         45.900000            13.850000          61.980000   
50%        1.000000         86.900000            17.170000         105.290000   
75%        1.000000        149.900000            24.040000         176.870000   
max       21.000000      13440.000000          1794.960000       13664.080000   

        freight_pct  
count  98666.000000  
mean       0.208804  
std        0.125749  
min        0.000000  
25%        0.116502  
50%        0.183256  
75%        0.275463  
max        0.955451  


In [15]:
orders.to_csv(f"{processed_path}/orders_clean.csv", index=False)
products.to_csv(f"{processed_path}/products_clean.csv", index=False)
geo_clean.to_csv(f"{processed_path}/geolocation_clean.csv", index=False)
payments.to_csv(f"{processed_path}/payments_clean.csv", index=False)
reviews.to_csv(f"{processed_path}/reviews_clean.csv", index=False)
order_items.to_csv(f"{processed_path}/order_items_clean.csv", index=False)
customers.to_csv(f"{processed_path}/customers_clean.csv", index=False)
sellers.to_csv(f"{processed_path}/sellers_clean.csv", index=False)
order_item_totals.to_csv(f"{processed_path}/order_item_totals.csv", index=False)

print("All cleaned tables saved to", processed_path)

All cleaned tables saved to ../data/processed
